# MuscleMap Thigh — Asian Dataset Evaluation (Thigh, Water)

Computes per-muscle Dice / Hausdorff metrics for MuscleMap **thigh** segmentations on the **MRI_data_asian** thigh water images.

- **Predictions**: `asian_segs_water/{subject}/Thigh/*_dseg*`
- **Ground truth**: `MRI_data_asian/MRI_data/{subject}/Thigh/mask_muscles.nii.gz` (labels 1–13)

**The Asian GT is unilateral** — each label covers only one side; the other side is background.  
L and R model predictions are therefore evaluated **separately** against the same GT label.  
The thigh model uses the 1–28 label scheme (Zenodo 19633000), distinct from the WB model's 7xxx scheme.  
`gluteus_maximus` (GT label 13) has no model label → all-zero prediction.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1

SEG_DIR    = os.path.join('..', 'asian_segs_water')
DATA_ROOT  = os.path.join('..', '..', 'MRI_data_asian', 'MRI_data')
RESULT_DIR = 'results_asian_water'
os.makedirs(RESULT_DIR, exist_ok=True)

# (muscle_name, asian_gt_label, mm_labels)
# GT is UNILATERAL — L and R evaluated separately against the same GT label.
# Thigh model 1-28 label scheme (Zenodo 19633000).
# biceps_femoris: Long Head + Short Head kept together per side.
# gluteus_maximus: not in model → None.
MUSCLES = [
    ('rectus_femoris_L',     1,  [7 ]),
    ('rectus_femoris_R',     1,  [8 ]),
    ('vastus_lateralis_L',   2,  [1 ]),
    ('vastus_lateralis_R',   2,  [2 ]),
    ('vastus_intermedius_L', 3,  [3 ]),
    ('vastus_intermedius_R', 3,  [4 ]),
    ('vastus_medialis_L',    4,  [5 ]),
    ('vastus_medialis_R',    4,  [6 ]),
    ('sartorius_L',          5,  [9 ]),
    ('sartorius_R',          5,  [10]),
    ('gracilis_L',           6,  [11]),
    ('gracilis_R',           6,  [12]),
    ('biceps_femoris_L',     7,  [17, 19]),  # long + short head, left
    ('biceps_femoris_R',     7,  [18, 20]),  # long + short head, right
    ('semitendinosus_L',     8,  [15]),
    ('semitendinosus_R',     8,  [16]),
    ('semimembranosus_L',    9,  [13]),
    ('semimembranosus_R',    9,  [14]),
    ('adductor_brevis_L',    10, [25]),
    ('adductor_brevis_R',    10, [26]),
    ('adductor_longus_L',    11, [23]),
    ('adductor_longus_R',    11, [24]),
    ('adductor_magnus_L',    12, [21]),
    ('adductor_magnus_R',    12, [22]),
    ('gluteus_maximus',      13, None),
]

print(f'SEG_DIR   : {os.path.abspath(SEG_DIR)}')
print(f'DATA_ROOT : {os.path.abspath(DATA_ROOT)}')
print(f'RESULT_DIR: {os.path.abspath(RESULT_DIR)}')

In [ ]:
# ── Discover Thigh segmentation files ─────────────────────────────────────────

seg_files = sorted(glob.glob(os.path.join(SEG_DIR, '*', 'Thigh', '*_dseg*')))
print(f'Found {len(seg_files)} Thigh segmentation files:')
for p in seg_files:
    print(' ', p)

In [ ]:
def evaluate_muscle(muscle_name, gt_label, mm_labels, seg_files, result_dir):
    """
    mm_labels: list of ints to OR into the prediction mask, or None (→ all-zero pred).
    """
    results = []
    for seg_path in seg_files:
        parts   = seg_path.replace('\\', '/').split('/')
        subject = parts[-3]

        gt_path = os.path.join(DATA_ROOT, subject, 'Thigh', 'mask_muscles.nii.gz')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        seg_sitk = sitk.ReadImage(seg_path)
        seg_raw  = sitk.GetArrayFromImage(seg_sitk)

        if mm_labels is not None:
            pred_arr = np.zeros_like(seg_raw, dtype=np.uint8)
            for lbl in mm_labels:
                pred_arr |= (seg_raw == lbl).astype(np.uint8)
        else:
            pred_arr = np.zeros_like(seg_raw, dtype=np.uint8)

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            hd = np.nan

        results.append({
            'subject':                              subject,
            'seg_file':                             os.path.basename(seg_path),
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df       = pd.DataFrame(results)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_musclemap_thigh_asian_water.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────

dfs = {}
for muscle_name, gt_label, mm_labels in MUSCLES:
    print(f'\n── {muscle_name}  (GT={gt_label}, MM={mm_labels}) ──')
    dfs[muscle_name] = evaluate_muscle(
        muscle_name, gt_label, mm_labels,
        seg_files, RESULT_DIR,
    )

print('\nDone.')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────

from IPython.display import display

summary_rows = []
for muscle_name, df in dfs.items():
    dice_col = f'{muscle_name}_dice'
    hd_col   = f'{muscle_name}_hausdorff'
    summary_rows.append({
        'muscle':         muscle_name,
        'n':              len(df),
        'dice_mean':      df[dice_col].mean(),
        'dice_std':       df[dice_col].std(),
        'hausdorff_mean': df[hd_col].mean(),
        'hausdorff_std':  df[hd_col].std(),
    })

summary = pd.DataFrame(summary_rows).set_index('muscle')
summary_path = os.path.join(RESULT_DIR, 'summary_musclemap_thigh_asian_water.csv')
summary.to_csv(summary_path)
print(f'Summary saved → {summary_path}\n')
display(summary.round(4))